# Seasonal Naive Baseline
This notebook imports reusable project code from `src/` and uses the real local M5 data when available.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
from src.config import settings
from src.data.loader import load_m5_data
from src.data.preprocessing import prepare_m5_long
prepared_path = settings.processed_dir / "m5_prepared.csv"
if prepared_path.exists():
    prepared = pd.read_csv(prepared_path, parse_dates=["date"])
else:
    data = load_m5_data()
    prepared = prepare_m5_long(data["sales"], data["calendar"], data["prices"])
print(prepared.shape)


In [ ]:
from src.forecasting.seasonal_naive import SeasonalNaive
from src.forecasting.trainer import chronological_split
train, valid = chronological_split(prepared, 28)
model = SeasonalNaive(7)
example_id = train["series_id"].astype(str).iloc[0]
h = train[train["series_id"].astype(str)==example_id].sort_values("date")
v = valid[valid["series_id"].astype(str)==example_id].sort_values("date")
forecast = model.forecast_values(h["demand"], len(v))
forecast[:10]

In [ ]:
from src.evaluation.metrics import mae
from src.evaluation.wmape import wmape
from src.evaluation.rmsse import rmsse
from src.evaluation.bias import forecast_bias
print({"MAE": mae(v["demand"], forecast), "WMAPE": wmape(v["demand"], forecast), "RMSSE": rmsse(v["demand"], forecast, h["demand"]), "Bias": forecast_bias(v["demand"], forecast)})